In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from constrerl.erl_schema import (
    entity_labels,
    relations,
    
)
from constrerl.annotator import Annotator, AnnotationTypes
from constrerl.annotation_model import (
    AnnotatedArticle,
    load_collection,
    
)

In [3]:
dev_articles=load_collection("Dev")
train_articles=load_collection("Train")

In [4]:
dev_articles

{'35766370': AnnotatedArticle(metadata=Metadata(title='Orthopedic Surgery Causes Gut Microbiome Dysbiosis and Intestinal Barrier Dysfunction in Prodromal Alzheimer Disease Patients: A Prospective Observational Cohort Study.', abstract='To investigate gut microbiota and intestinal barrier function changes after orthopedic surgery in elderly patients with either normal cognition (NC) or a prodromal Alzheimer disease phenotype (pAD) comprising either subjective cognitive decline (SCD) or amnestic mild cognitive impairment (aMCI). Homeostatic disturbances induced by surgical trauma and/or stress can potentially alter the gut microbiota and intestinal barrier function in elderly patients before and after orthopedic surgery. In this prospective cohort study, 135 patients were subject to preoperative neuropsychological assessment and then classified into: NC (n=40), SCD (n=58), or aMCI (n=37). Their gut microbiota, bacterial endotoxin (lipopolysaccharide), tight junction (TJ) protein, and inf

In [5]:
test_article=dev_articles[list(dev_articles.keys())[0]]
test_article

AnnotatedArticle(metadata=Metadata(title='Orthopedic Surgery Causes Gut Microbiome Dysbiosis and Intestinal Barrier Dysfunction in Prodromal Alzheimer Disease Patients: A Prospective Observational Cohort Study.', abstract='To investigate gut microbiota and intestinal barrier function changes after orthopedic surgery in elderly patients with either normal cognition (NC) or a prodromal Alzheimer disease phenotype (pAD) comprising either subjective cognitive decline (SCD) or amnestic mild cognitive impairment (aMCI). Homeostatic disturbances induced by surgical trauma and/or stress can potentially alter the gut microbiota and intestinal barrier function in elderly patients before and after orthopedic surgery. In this prospective cohort study, 135 patients were subject to preoperative neuropsychological assessment and then classified into: NC (n=40), SCD (n=58), or aMCI (n=37). Their gut microbiota, bacterial endotoxin (lipopolysaccharide), tight junction (TJ) protein, and inflammatory cyt

In [6]:
test_article.metadata.abstract

'To investigate gut microbiota and intestinal barrier function changes after orthopedic surgery in elderly patients with either normal cognition (NC) or a prodromal Alzheimer disease phenotype (pAD) comprising either subjective cognitive decline (SCD) or amnestic mild cognitive impairment (aMCI). Homeostatic disturbances induced by surgical trauma and/or stress can potentially alter the gut microbiota and intestinal barrier function in elderly patients before and after orthopedic surgery. In this prospective cohort study, 135 patients were subject to preoperative neuropsychological assessment and then classified into: NC (n=40), SCD (n=58), or aMCI (n=37). Their gut microbiota, bacterial endotoxin (lipopolysaccharide), tight junction (TJ) protein, and inflammatory cytokines in blood were measured before surgery and on postsurgical day 1, 3, and 7 (or before discharge). The short-chain fatty acid (SCFA)-producing bacteria were lower while the gram-negative bacteria, lipopolysaccharide a

In [7]:
from constrerl.annotator import extract_noun_phrases
print(extract_noun_phrases(test_article.metadata.abstract), test_article.metadata.abstract)

{'gut microbiota and intestinal barrier function changes': AnnotationSpan(start_idx=15, end_idx=69, text='gut microbiota and intestinal barrier function changes'), 'orthopedic surgery': AnnotationSpan(start_idx=473, end_idx=491, text='orthopedic surgery'), 'elderly patients': AnnotationSpan(start_idx=439, end_idx=455, text='elderly patients'), 'either normal cognition': AnnotationSpan(start_idx=120, end_idx=143, text='either normal cognition'), 'NC': AnnotationSpan(start_idx=625, end_idx=627, text='NC'), 'pAD': AnnotationSpan(start_idx=1058, end_idx=1061, text='pAD'), 'either subjective cognitive decline': AnnotationSpan(start_idx=209, end_idx=244, text='either subjective cognitive decline'), 'SCD': AnnotationSpan(start_idx=636, end_idx=639, text='SCD'), 'amnestic mild cognitive impairment': AnnotationSpan(start_idx=254, end_idx=288, text='amnestic mild cognitive impairment'), 'aMCI': AnnotationSpan(start_idx=651, end_idx=655, text='aMCI'), 'Homeostatic disturbances': AnnotationSpan(st

In [8]:
# model = Llama.from_pretrained(
#     "NousResearch/Hermes-3-Llama-3.2-3B-GGUF",
#     filename="*.Q8_0.gguf",
#     n_gpu_layers=-1,
#     n_ctx=8096,
#     temperature=0.1,
# )

In [9]:
from constrerl.beam_search.grammar import GBNF_PARSER, test_against_grammar

In [10]:
test_grammar = """
root ::= ent-list
entity ::= entity-type" ("entity-str")"
arbitrary-str ::= (([0-9a-fA-F]|" "){1, 4})
entity-type ::= "Anatomical Location"|"Animal"|"Biomedical Technique"|"Bacteria"|"Chemical"|"Dietary Supplement"|"DDF"|"Drug"|"Food"|"Gene"|"Human"|"Microbiome"|"Statistical Technique"
entity-str ::= "Orthopedic Surgery"|"Gut Microbiome Dysbiosis"|"Intestinal Barrier Dysfunction"|"Prodromal Alzheimer Disease Patients"|"Prospective Observational Cohort Study"
ent-list ::= entity ("\\n" entity)*
"""
test_grammar_parsed = GBNF_PARSER.parse(test_grammar)
print(test_grammar_parsed.pretty())

start
  rule
    identifier	root
    expression
      term
        factor
          primary
            identifier	ent-list
  rule
    identifier	entity
    expression
      term
        factor
          primary
            identifier	entity-type
        factor
          primary
            literal	" ("
        factor
          primary
            identifier	entity-str
        factor
          primary
            literal	")"
  rule
    identifier	arbitrary-str
    expression
      term
        factor
          primary
            expression
              term
                factor
                  primary
                    expression
                      term
                        factor
                          primary
                            char_class
                              0
                              -
                              9
                              a
                              -
                              f
                              A

In [11]:
test_against_grammar("DDF", test_grammar_parsed)


(True, 3)

In [12]:
from llama_cpp import Llama

# model_path = "quants/hermes-3-2-3B-lora-entities.gguf"
model_path = "quants/hermes-3-2-3B.gguf"
model = Llama(
    model_path,
    n_gpu_layers=-1,
    n_ctx=4096,
    logits_all=True,
    verbose=False
    # draft_model=LlamaPromptLookupDecoding(num_pred_tokens=10),
)

llama_context: n_ctx_seq (4096) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


In [13]:
entities=[lbl["label"] for lbl in entity_labels]
entities

['Anatomical Location',
 'Animal',
 'Biomedical Technique',
 'Bacteria',
 'Chemical',
 'Dietary Supplement',
 'DDF',
 'Drug',
 'Food',
 'Gene',
 'Human',
 'Microbiome',
 'Statistical Technique']

In [14]:
test_articles = {"test": test_article.metadata}

In [15]:
# annotator_train = AnnotatorHelper(
#     gen_tokens=512,
#     top_k=5,
#     add_rag=True
# )
# annotator_train.model = model
# annotator_train.load_articles(train_articles)
# annotator_train.save_articles(Path("./data/annotations/prepared_train.json"))

In [16]:
dev_articles_reduced = {k: v for i, (k, v) in enumerate(dev_articles.items()) if i < 1}
# dev_articles_reduced = dev_articles #{k: v for i, (k, v) in enumerate(dev_articles.items()) if i < 1}

In [20]:
from constrerl.annotator import AnnotatorHelper, BeamSearchConfig
from pathlib import Path


annotator = AnnotatorHelper(
    gen_tokens=4,
    top_k=5,
    naive_annotations=True,
    naive_only=False,
    # add_rag=True,
    beam_search=BeamSearchConfig(top_k=2, max_depth=3),
)
annotator.model = model
# annotator.load_articles(dev_articles | train_articles)
# annotator.save_articles(Path("./data/annotations/prepared_dev_train.json"))

annotator.load_articles_from_path(Path("./data/annotations/prepared_dev.json"))
annotator.load_concepts(Path("./data/Annotations/uri_collection_concepts.json"))

In [ ]:
annotated_articles = annotator.annotate(
    {
        id: article.metadata for id, article in dev_articles_reduced.items()
    },  
    annotate=[AnnotationTypes.ENTITY]
)

Annotating articles:   0%|          | 0/1 [00:00<?, ?it/s]

No valid tokens found at reached at depth=6  on str self.new_str='Anatomical Location (In' resp txt fer, setting to filtered_logits to filtered_logits={} (from logits logits=[{42861: np.float32(-1.8778895), 45864: np.float32(-2.761137), 38818: np.float32(-2.8202744), 279: np.float32(-3.1535015), 77: np.float32(-3.178568), 809: np.float32(-3.4260073), 12821: np.float32(-3.4765158), 13421: np.float32(-3.7646637), 902: np.float32(-3.8634567), 2542: np.float32(-3.8696241), 78134: np.float32(-4.0670176), 1148: np.float32(-4.266632), 2221: np.float32(-4.3021793), 41294: np.float32(-4.3496714), 14285: np.float32(-4.353072), 376: np.float32(-4.41815), 10015: np.float32(-4.427703), 90927: np.float32(-4.4989805), 2251: np.float32(-4.5716066), 7114: np.float32(-4.635022), 3878: np.float32(-4.6664276), 4339: np.float32(-4.6972847), 6498: np.float32(-4.748945), 744: np.float32(-4.767049), 37618: np.float32(-4.8948336), 10663: np.float32(-5.0448513), 5862: np.float32(-5.0740023), 3836: np.float32(-5

Finish reason stop reached at depth=9 with tokens=[128000, 128006, 9125, 128007, 271, 2675, 527, 264, 6593, 6335, 37142, 1113, 264, 6593, 12624, 2316, 323, 8278, 13, 128009, 128006, 882, 128007, 271, 67637, 16771, 292, 48190, 74105, 52683, 18654, 8385, 638, 65142, 70728, 285, 323, 1357, 65050, 72087, 88266, 304, 72372, 442, 278, 44531, 31974, 44430, 25, 362, 32134, 9262, 31943, 1697, 84675, 371, 19723, 13, 128009, 128006, 78191, 128007, 271, 35, 81, 773, 320, 67637, 16771, 292, 48190, 8] and self.new_str='Drug (Orthopedic Surgery)', generated response: {'id': 'cmpl-b707b9c0-4be5-4270-b526-692e9066e50b', 'object': 'text_completion', 'created': 1777734269, 'model': 'quants/hermes-3-2-3B.gguf', 'choices': [{'text': '', 'index': 0, 'logprobs': {'tokens': [], 'text_offset': [], 'token_logprobs': [], 'top_logprobs': [], 'top_logprob_tokens': [], 'all_logprobs': array([[-15.236291, -18.398024, -15.793058, ..., -10.436089, -10.43708 ,
        -10.439043]], shape=(1, 128256), dtype=float32)}, '

Finish reason stop reached at depth=10 with tokens=[128000, 128006, 9125, 128007, 271, 2675, 527, 264, 6593, 6335, 37142, 1113, 264, 6593, 12624, 2316, 323, 8278, 13, 128009, 128006, 882, 128007, 271, 67637, 16771, 292, 48190, 74105, 52683, 18654, 8385, 638, 65142, 70728, 285, 323, 1357, 65050, 72087, 88266, 304, 72372, 442, 278, 44531, 31974, 44430, 25, 362, 32134, 9262, 31943, 1697, 84675, 371, 19723, 13, 128009, 128006, 78191, 128007, 271, 35, 81, 773, 320, 67637, 16771, 292, 48190, 8, 128039, 39, 7282, 320, 42379, 442, 278, 44531, 31974, 44430, 8] and self.new_str='Human (Prodromal Alzheimer Disease Patients)', generated response: {'id': 'cmpl-37593e3c-ca0f-48f3-bb64-999be954bd7d', 'object': 'text_completion', 'created': 1777734301, 'model': 'quants/hermes-3-2-3B.gguf', 'choices': [{'text': '', 'index': 0, 'logprobs': {'tokens': [], 'text_offset': [], 'token_logprobs': [], 'top_logprobs': [], 'top_logprob_tokens': [], 'all_logprobs': array([[-18.702385, -22.200832, -19.498962, ...,

Finish reason stop reached at depth=8 with tokens=[128000, 128006, 9125, 128007, 271, 2675, 527, 264, 6593, 6335, 37142, 1113, 264, 6593, 12624, 2316, 323, 8278, 13, 128009, 128006, 882, 128007, 271, 67637, 16771, 292, 48190, 74105, 52683, 18654, 8385, 638, 65142, 70728, 285, 323, 1357, 65050, 72087, 88266, 304, 72372, 442, 278, 44531, 31974, 44430, 25, 362, 32134, 9262, 31943, 1697, 84675, 371, 19723, 13, 128009, 128006, 78191, 128007, 271, 35, 81, 773, 320, 67637, 16771, 292, 48190, 8, 128039, 33, 78852, 320, 38, 332, 18654, 8385, 638, 65142, 70728, 285, 8, 128039, 38, 1994, 320, 1090, 65050, 72087, 88266, 8] and self.new_str='Gene (Intestinal Barrier Dysfunction)', generated response: {'id': 'cmpl-bd50c254-7c01-4f3a-b46f-489344c2f307', 'object': 'text_completion', 'created': 1777734331, 'model': 'quants/hermes-3-2-3B.gguf', 'choices': [{'text': '', 'index': 0, 'logprobs': {'tokens': [], 'text_offset': [], 'token_logprobs': [], 'top_logprobs': [], 'top_logprob_tokens': [], 'all_logpr

...

Finish reason stop reached at depth=8 with tokens=[128000, 128006, 9125, 128007, 271, 2675, 527, 264, 6593, 6335, 37142, 1113, 264, 6593, 12624, 2316, 323, 8278, 13, 128009, 128006, 882, 128007, 271, 1271, 19874, 18340, 53499, 6217, 323, 63900, 22881, 734, 4442, 1306, 30299, 16771, 292, 15173, 304, 29920, 6978, 449, 3060, 4725, 75310, 320, 10153, 8, 477, 264, 14814, 442, 278, 44531, 8624, 82423, 320, 79, 1846, 8, 46338, 3060, 44122, 25702, 18174, 320, 3624, 35, 8, 477, 1097, 77, 10027, 23900, 25702, 53317, 320, 64, 44, 11487, 8, 128009, 128006, 78191, 128007, 271, 2127, 6756, 64, 75, 10067, 320, 68, 275, 383, 81, 4725, 75310, 8, 128039, 2127, 22612, 950, 10067, 320, 50998, 4725, 75310, 8, 128039, 2127, 22612, 950, 10067, 320, 50998, 4725, 75310, 8, 128039, 38, 268, 68, 320, 50998, 4725, 75310, 8] and self.new_str='Gene (either normal cognition)', generated response: {'id': 'cmpl-0b5d000c-a360-407b-8d13-5fa154c9d6ea', 'object': 'text_completion', 'created': 1777734458, 'model': 'quants/

No valid tokens found at reached at depth=2  on str self.new_str='Ana' resp txt esthesia, setting to filtered_logits to filtered_logits={} (from logits logits=[{63623: np.float32(-0.73324275), 54120: np.float32(-2.2801406), 478: np.float32(-2.4687002), 71109: np.float32(-2.690624), 22689: np.float32(-3.1287773), 21215: np.float32(-3.7960384), 386: np.float32(-4.8860836), 11: np.float32(-5.0015163), 23880: np.float32(-5.110697), 480: np.float32(-5.3196793), 64853: np.float32(-5.4629316), 13030: np.float32(-5.6556416), 38672: np.float32(-5.682687), 80274: np.float32(-5.7064133), 261: np.float32(-5.7075577), 432: np.float32(-5.8319225), 85744: np.float32(-5.836878), 362: np.float32(-5.938137), 8274: np.float32(-5.955534), 328: np.float32(-5.981079), 445: np.float32(-5.988535), 393: np.float32(-6.004303), 25: np.float32(-6.0551567), 356: np.float32(-6.061632), 735: np.float32(-6.1022177), 772: np.float32(-6.123337), 423: np.float32(-6.189643), 83305: np.float32(-6.2792416), 34297: np.float

No valid tokens found at reached at depth=1  on str self.new_str='S' resp txt urgical, setting to filtered_logits to filtered_logits={} (from logits logits=[{57673: np.float32(-0.2775801), 85392: np.float32(-1.6461638), 54376: np.float32(-4.80019), 5673: np.float32(-5.46739), 86255: np.float32(-6.567587), 3884: np.float32(-6.646509), 7270: np.float32(-6.7992363), 946: np.float32(-6.870775), 39095: np.float32(-6.929882), 16: np.float32(-7.025276), 43216: np.float32(-7.1026077), 5169: np.float32(-7.251297), 82149: np.float32(-7.35962), 25: np.float32(-7.3613644), 8362: np.float32(-7.5044127), 55704: np.float32(-7.5776253), 91064: np.float32(-7.6064816), 2871: np.float32(-7.7037487), 729: np.float32(-7.7672615), 2168: np.float32(-7.823386), 25100: np.float32(-7.867773), 48937: np.float32(-7.93484), 2599: np.float32(-7.9721947), 773: np.float32(-8.079625), 5308: np.float32(-8.113367), 425: np.float32(-8.282597), 47876: np.float32(-8.299833), 87191: np.float32(-8.340514), 399: np.float32(-8

Reached end token self.cfg.end_token='\n' or EOS in child.new_str='Gene (surgical trauma)\n'
Decoded beam search output: added_str='Anatomical Location (orthopedic surgery)\n' skip_tokens=None t_nd.new_str='Anatomical Location (orthopedic surgery)\n'
Reached end token self.cfg.end_token='\n' or EOS in child.new_str='Anatomical Location (gut microbiota)\n'
Reached end token self.cfg.end_token='\n' or EOS in child.new_str='Anatomical Location (gut microbiota)\n'
Reached end token self.cfg.end_token='\n' or EOS in child.new_str='Dietary Supplement (gut microbiota)\n'
Reached end token self.cfg.end_token='\n' or EOS in child.new_str='Drug (surgical trauma)\n'
Reached end token self.cfg.end_token='\n' or EOS in child.new_str='Microbiome (gut microbiota)\n'
Reached end token self.cfg.end_token='\n' or EOS in child.new_str='Microbiome (gut microbiota)\n'


Reached end token self.cfg.end_token='\n' or EOS in child.new_str='Statistical Technique (surgical trauma)\n'
Decoded beam search output: added_str='Anatomical Location (gut microbiota)\n' skip_tokens=None t_nd.new_str='Anatomical Location (gut microbiota)\n'


In [91]:
annotator.model._scores

array([], shape=(0, 128256), dtype=float32)

In [ ]:
annotated_articles

{'35766370': AnnotatedArticle(metadata=Metadata(title='Orthopedic Surgery Causes Gut Microbiome Dysbiosis and Intestinal Barrier Dysfunction in Prodromal Alzheimer Disease Patients: A Prospective Observational Cohort Study.', author='Fangyan Liu; Mei Duan; Huiqun Fu; Guoguang Zhao; Ying Han; Fei Lan; Zara Ahmed; Guanglei Cao; Zheng Li; Daqing Ma; Tianlong Wang', journal='Annals of surgery', year=2022, abstract='To investigate gut microbiota and intestinal barrier function changes after orthopedic surgery in elderly patients with either normal cognition (NC) or a prodromal Alzheimer disease phenotype (pAD) comprising either subjective cognitive decline (SCD) or amnestic mild cognitive impairment (aMCI). Homeostatic disturbances induced by surgical trauma and/or stress can potentially alter the gut microbiota and intestinal barrier function in elderly patients before and after orthopedic surgery. In this prospective cohort study, 135 patients were subject to preoperative neuropsychologic

In [ ]:
annotated_articles[list(annotated_articles.keys())[0]].entities

[Entity(start_idx=117, end_idx=124, location='title', text_span='Patients', label='human', uri='http://id.nlm.nih.gov/mesh/D010361'),
 Entity(start_idx=89, end_idx=124, location='title', text_span='Prodromal Alzheimer Disease Patients', label='human', uri='http://id.nlm.nih.gov/mesh/D010361'),
 Entity(start_idx=26, end_idx=39, location='title', text_span='Gut Microbiome', label='microbiome', uri='http://purl.obolibrary.org/obo/NCIT_C14329'),
 Entity(start_idx=30, end_idx=39, location='title', text_span='Microbiome', label='microbiome', uri='http://purl.obolibrary.org/obo/NCIT_C14329'),
 Entity(start_idx=99, end_idx=107, location='title', text_span='Alzheimer', label='DDF', uri='http://purl.obolibrary.org/obo/NCIT_C2866'),
 Entity(start_idx=99, end_idx=115, location='title', text_span='Alzheimer Disease', label='DDF', uri='http://purl.obolibrary.org/obo/NCIT_C2866'),
 Entity(start_idx=26, end_idx=28, location='title', text_span='Gut', label='anatomical location', uri='http://purl.obolib

In [ ]:
# annotator.add_concept_uris(annotated_articles)

In [ ]:
from constrerl.utils import prepare_for_eval

In [ ]:
from constrerl.eval.evaluate import (
    eval_submission_mention_level_RE,
    eval_submission_NER,
    eval_submission_NERD,
    eval_submission_concept_level_RE,
)


def tuple_to_scores(tup):
    # precision, recall, f1, micro_precision, micro_recall, micro_f1
    return {
        "precision": tup[0],
        "recall": tup[1],
        "f1": tup[2],
        "micro_precision": tup[3],
        "micro_recall": tup[4],
        "micro_f1": tup[5],
    }


# ground_truth_test = {"test": test_article}
eval_functions = {
    "NER": eval_submission_NER,
    "NERD": eval_submission_NERD,
    "mention_level_RE": eval_submission_mention_level_RE,
    "concept_level_RE": eval_submission_concept_level_RE,
}
scores_list = []
for typ, eval_fun in eval_functions.items():
    print(f"Evaluating {typ}...")
    scores = eval_fun(
        prepare_for_eval(annotated_articles), prepare_for_eval(dev_articles_reduced)
    )
    scores=tuple_to_scores(scores)
    scores["type"] = typ
    scores_list.append(scores)

import pandas as pd
df_scores = pd.DataFrame(scores_list)
df_scores


Evaluating NER...
=== Removed 6476 duplicated entities from predictions ===
=== Removed 4145 overlapping entities ===
Evaluating NERD...
=== Removed 6476 duplicated entities from predictions ===
=== Removed 4145 overlapping entities ===
Evaluating mention_level_RE...
Evaluating concept_level_RE...


,precision,recall,f1,micro_precision,micro_recall,micro_f1,type
0,0.494921,0.790845,0.599543,0.508262,0.805236,0.623177,NER
1,0.429336,0.681450,0.518440,0.420130,0.665609,0.515119,NERD
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,mention_level_RE
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,concept_level_RE
